# Who Keeps Philadelphia Running After Dark?

**Technical Documentation**

This document records the full analysis pipeline behind the project: where Philadelphia's night-shift
commuters live and work, how easily they reach overnight (Owl) bus service on foot, how much of that walk is
poorly lit, and how cuts to Owl service would change both. Each numbered section from 5 onward corresponds to
one step notebook in `notebooks/`; reusable calculations live in `src/`, and every parameter is set in `config.py`.

## 1. Introduction

This section states the research question, gives an overview of the pipeline, and lists the conventions that
every step follows.

### 1.1 Research Question

Who works at night in Philadelphia, how long does it take them to reach overnight bus service, and how much of
that walk is on poorly lit streets? We then simulate three Owl service reductions (doubled headways, removal of
the B1 OWL, removal of the L1 OWL) and test whether the burden falls disproportionately on car-free
neighborhoods. Two outcome dimensions are kept separate throughout: **efficiency** = walking time + schedule
delay; **safety** = poorly lit walking distance + stop lighting (waiting time is not part of safety).

### 1.2 Pipeline Overview

The pipeline runs from raw data to website exports in eleven layers. Steps 00, 01 and 03 are independent and can
run in parallel; step 05a needs only the network and stops, so it can run before 04.

```
L0 Data acquisition ─┐
L1 Data cleaning ────┼─> L2 Walk network (00)
                     │        │
                     ├─> L3 Commuter profile (01) ──> L4 Placing commuters (02)
                     │                                   │
                     └─> L5 Night service (03) ──────────┤
                              │                          v
                              ├──> L6 Access / coverage (04) ──> L6b OD connectivity (04b)
                              v                          │                │
                         L7a Lighting (05a) ──> L7b Corridors (05b)       │
                                                         │                │
                                                         v                │
                                          L8 Service-reduction scenarios (06) <──┘
                                                │                 │
                                                v                 v
                                   L9 Association test (06b)   L10 Sensitivity (06c)
                                                │                 │
                                                └──> L11 Web export (data/web/)
```

| Layer | Section | Notebook | `src/` module |
| --- | --- | --- | --- |
| L0–L1 Acquisition & cleaning | 2–4 | start of each notebook | `fetch` |
| L2 Walk network | 5 | `00_network` | `network` |
| L3 Commuter profile | 6 | `01_profile` | `pums` |
| L4 Placing commuters | 7 | `02_place_on_network` | `pums`, `network` |
| L5 Night service | 8 | `03_night_service` | `gtfs` |
| L6 Access & coverage | 9 | `04_access` | `access` |
| L6b OD connectivity | 10 | `04b_od_check` | `od` |
| L7a Lighting | 11 | `05a_lighting` | `lighting` |
| L7b Corridors | 12 | `05b_corridors` | `corridors` |
| L8 Scenarios | 13 | `06_scenarios` | `scenarios` |
| L9 Association test | 14 | `06b_regression` | `regression` |
| L10 Sensitivity | 15 | `06c_sensitivity` | `sensitivity` |
| L11 Web export | 16 | end of `06_scenarios` | — |

### 1.3 Conventions

A few rules keep every step re-runnable on its own and keep units consistent. They are summarised in the block
below for reference; the parameters themselves are enforced in `config.py`.

In [ ]:
# Project-wide conventions (reference only; nothing to run)
#
# File handoff
#   - Each notebook reads only data/raw/ and data/processed/, never another notebook's variables,
#     so any step can be re-run on its own.
#   - Outputs carry the step prefix (e.g. 02_home_nodes.parquet) and are overwritten on re-run.
#
# Functions, not globals
#   - Work-end weighting (step 02), stop/route choice (04), corridor tracing (05b) and the regression (06b)
#     live in src/ and take parameters as arguments, because steps 06 and 06c call them in loops.
#
# CRS and units
#   - Projected CRS: EPSG:2272 (US survey feet). Convert metres with config.m_to_ft() before any buffer
#     or spatial join.
#   - Network distances are always in metres.
#
# Storage formats
#   - data/raw/:       original downloads; written once, never edited.
#   - data/processed/: GeoParquet (EPSG:2272) for tables and geometry; CSV for small summaries
#                      (profile, odds ratios); JSON for scalar parameters (p_evening, car-free multiplier).
#   - data/web/:       EPSG:4326, display fields only, simplified geometry.
#   - GEOID, stop_id and route_id are always stored as strings.
#   - Scenario results are one long table (scenario × end × node_id), not one file per scenario.
#
# Version control
#   - Commit data/processed/ and data/web/; commit raw files under 50 MB.
#   - Larger raw files are re-downloaded with src/fetch.py (documented in README).

### 1.4 Environment Setup

Create the conda environment from `environment.yml` (pandana installs most reliably from conda-forge) and copy
`.env.example` to `.env` with a Census API key. The block below puts the repository root on the import path and
loads the project parameters.

In [ ]:
# Put the repo root on the import path and load project parameters and helper modules
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import config
from config import OUT

# import numpy as np
# import pandas as pd
# import geopandas as gpd
# from src import fetch, pums, network, gtfs, access, lighting, corridors, scenarios

## 2. Data Acquisition

Each source has one `fetch_xxx()` function in `src/fetch.py` that saves the download unchanged to `data/raw/`;
no cleaning happens here. Variable codes, dataset names and field names noted as "verify" are checked against the
source before the first pull, and any mismatch with the project definitions is reported rather than silently
changed.

In [ ]:
# --- ACS 5-year 2020–2024, Philadelphia tracts (state 42, county 101) ---
# B08302 time leaving home to go to work: _001 total, _002 12:00–4:59 a.m., _015 4:00–11:59 p.m.
# B08141 means of transportation by vehicles available: _001 total, _002 no vehicle available
# Verify variable codes against the Census API variable list before pulling; keep MOE columns.
# fetch.fetch_acs_tracts()

In [ ]:
# --- 2020 Decennial Census block population (PL 94-171) ---
# Total population per block; used to push tract counts down to blocks in step 02.
# fetch.fetch_block_population()

In [ ]:
# --- TIGER/Line geometries: tracts, blocks, PUMAs, city boundary ---
# Used for the walk-network extent (step 00) and tract → block → node allocation (step 02).
# fetch.fetch_tiger()   # to add to src/fetch.py

In [ ]:
# --- PUMS 2020–2024 5-year microdata, by state + PUMA (states as fixed in the engineering record) ---
# Person:  JWDP JWAP JWTRNS NAICSP PINCP WAGP SEX AGEP RAC1P HISP PUMA POWPUMA PWGTP PWGTP1–80 SERIALNO
# Housing: HINCP VEH WGTP SERIALNO
# fetch.fetch_pums()

In [ ]:
# --- LODES Workplace Area Characteristics (WAC), Pennsylvania, latest year, JT00 (all jobs) ---
# Block × CNS01–CNS20 job counts; work-end input for step 02.
# fetch.fetch_lodes_wac(year=config.LODES_YEAR)

In [ ]:
# --- LODES Origin–Destination (OD), Pennsylvania, JT00 (all jobs, same as WAC), same year as WAC ---
# main file only: home and work both in PA; fields h_geocode, w_geocode, S000, SI01, SI02, SI03.
# The aux file (home outside PA) is not pulled: main + aux = WAC for every PA work block, so the
# out-of-state share of Philadelphia jobs is WAC − main. NJ and DE files are not pulled either,
# so Philadelphia residents working out of state are not counted.
# fetch.fetch_lodes_od(year=config.LODES_YEAR, job_type=config.OD_JOB_TYPE)   # to add to src/fetch.py

In [ ]:
# --- SEPTA bus GTFS feed ---
# Keep stops, routes, trips, stop_times, calendar, calendar_dates.
# fetch.fetch_gtfs()

In [ ]:
# --- OpenStreetMap walk network (OSMnx, network_type="walk") ---
# Downloaded in step 00 for the city boundary buffered by ~1.6 km (see Section 5).
# nodes, edges = network.build_walk_network()

In [ ]:
# --- Streetlight locations (OpenDataPhilly / ArcGIS REST, "street poles"; verify dataset name) ---
# Point layer of poles; input for effective lighting density in step 05a.
# fetch.fetch_streetlights()

In [ ]:
# --- 311 streetlight outage requests (Carto API), 2025-11-01 to 2026-10-31 ---
# Keep requested / closed timestamps, coordinates, status and closure reason; verify category name.
# fetch.fetch_311_outages()

In [ ]:
# --- ATUS non-daytime work shares by industry (entered by hand) ---
# BLS USDL-26-1533 Table 7 (2024–2025 averages, main); 2017–2018 values (USDL-19-1691, as used in Plotts 2025
# Appendix A.1) kept as a second column; BLS major groups mapped to CNS; suppressed groups set to 0 and flagged.
# Used only as an alternative work-end proxy in the sensitivity analysis (step 06c).
# atus = pd.read_csv(config.RAW / "atus" / "atus_night_share.csv", dtype={"CNS": str})

## 3. Data Characteristics

Before cleaning, each raw dataset is profiled: spatial unit, time period, record count, key fields, coverage of
Philadelphia, missing values and, for survey data, margins of error. These summaries document the inputs for
readers of the website and explain the cleaning choices made in Section 4.

In [ ]:
# Helper: one summary row per dataset (unit, period, n records, n fields, % missing in key fields)
# def describe_dataset(df, name, unit, period, key_cols):
#     return {"dataset": name, "unit": unit, "period": period, "n_records": len(df),
#             "n_fields": df.shape[1], "pct_missing_key": df[key_cols].isna().mean().max() * 100}
# summary_rows = []

### 3.1 Census and Survey Data

ACS summary tables give tract-level departure times and vehicle availability, with margins of error that are
often large for small tracts. The 2020 block counts supply the population weights for tract → block allocation.
PUMS microdata add individual detail (departure and arrival time, industry, income, vehicles) but are
geocoded only to PUMAs.

In [ ]:
# --- ACS: number of tracts, value ranges, and reliability (CV = MOE / 1.645 / estimate) per variable ---
# acs = pd.read_csv(config.RAW / "acs" / "...", dtype={"GEOID": str})
# cv = acs[moe_cols].values / 1.645 / acs[est_cols].values
# Report tracts with zero workers and the share of tracts with CV > 40% for B08302_002.

In [ ]:
# --- 2020 blocks: number of blocks, share with zero population, population distribution ---
# blocks_pop = pd.read_csv(config.RAW / "census_blocks" / "...", dtype={"GEOID": str})

In [ ]:
# --- PUMS: unweighted records vs weighted totals, Philadelphia PUMAs, valid JWDP/JWAP share ---
# Also report the unweighted sample size of night workers (by residence PUMA and by POWPUMA),
# since it limits how finely industry shares can be estimated.

### 3.2 Employment Data (LODES WAC and OD)

LODES WAC counts jobs at the workplace block for 20 NAICS-based sectors (CNS01–CNS20); industry (CNS) and earnings
(CE) are separate marginals and cannot be cross-tabulated. LODES OD links home and work blocks but reports industry
only in three broad groups (SI01–SI03), and its home locations are noise-infused synthetic data, so OD results are
reported at tract level only.

In [ ]:
# --- LODES: Philadelphia blocks with jobs, total jobs, jobs by CNS sector (bar chart) ---
# wac = pd.read_csv(config.RAW / "lodes" / "...", dtype={"w_geocode": str})
# wac.filter(like="CNS").sum().sort_values().plot.barh()

In [ ]:
# --- LODES: spatial concentration of jobs (map of jobs per block; share of jobs in the top 1% of blocks) ---

In [ ]:
# --- LODES OD: number of home–work block pairs, share with home and work both in Philadelphia,
#     jobs by SI group, and OD vs WAC job totals for Philadelphia work blocks ---

### 3.3 Transit Data (SEPTA GTFS)

The GTFS feed describes scheduled service through stops, routes, trips and stop times, with calendar rules for
each service day. Trips after midnight are written with times past 24:00 and belong to the previous service day.

In [ ]:
# --- GTFS: feed validity dates, number of routes / stops / trips ---

In [ ]:
# --- GTFS: departures by hour on the representative weekday, highlighting the 00:00–05:00 window ---

In [ ]:
# --- GTFS: route_short_name values containing "OWL"; confirm how B1 OWL and L1 OWL are written ---
# routes[routes.route_short_name.str.contains("OWL", case=False)]

### 3.4 Street Network and Geography (OSM, TIGER)

The OSM walk network is checked for size, edge-length distribution and connected components; TIGER geometries are
checked for counts and GEOID agreement with the ACS and LODES tables.

In [ ]:
# --- OSM: node and edge counts, edge length distribution (m), number and size of connected components ---

In [ ]:
# --- TIGER: tract / block / PUMA counts; GEOIDs that fail to match ACS, block population or LODES ---

### 3.5 Street Lighting Data (Streetlights, 311 Outages)

The streetlight layer shows where poles stand; 311 requests show when lights were reported out and when the cases
were closed. A closed case does not always mean the light was repaired, so status and closure reasons are
profiled here.

In [ ]:
# --- Streetlights: pole count, fields, CRS, duplicate points, poles outside the city boundary ---

In [ ]:
# --- 311: requests by category, status / closure-reason breakdown, share still open at window end ---

In [ ]:
# --- 311: distribution of report-to-close duration and monthly request counts over the outage window ---

### 3.6 Reference Shares (ATUS)

ATUS gives national shares of non-daytime work by industry. They are used only as an alternative work-end proxy,
so only spatial patterns and rankings are compared, never totals.

In [ ]:
# --- ATUS: show the CNS → share table; list sectors with no published value (set to 0 and flagged) ---

## 4. Data Cleaning

Cleaning happens inside the notebook of the step that uses each source; `data/raw/` is never modified. Each block
below names the source and the rule applied, with the known pitfall noted in the comment.

In [ ]:
# --- PUMS: merge person and housing records on SERIALNO ---
# pums = person.merge(housing, on="SERIALNO", how="left")

In [ ]:
# --- PUMS: decode JWDP / JWAP range codes into start and end times ---
# JWDP and JWAP are banded codes; map each band to its time range using the data dictionary,
# then flag overnight (00:00–04:59) and evening (16:00–21:00) bands.
# night_codes = pums.night_departure_codes(jwdp_dict)
# evening_codes = pums.evening_departure_codes(jwdp_dict)

In [ ]:
# --- PUMS: map NAICSP to LODES CNS sectors (Appendix B) ---
# Use the first two digits; look up alphanumeric codes (3MS, 4MS, ...) one by one in the dictionary;
# drop codes with no CNS equivalent (e.g. military).

In [ ]:
# --- PUMS: weights and standard errors ---
# Persons weighted by PWGTP, households by WGTP; standard errors from the 80 replicate weights.
# Denominators differ by end:
#   home end = commuters with a reported departure time
#   work end = all workers in industry k with POWPUMA in Philadelphia, including work-from-home (JWTRNS = 11)
# pums.weighted_share(...); pums.replicate_se(...)

In [ ]:
# --- ACS: GEOID as string; keep MOE columns ---
# B08302 and B08141 cannot be cross-tabulated; they are only compared as an overlap (bivariate) layer.

In [ ]:
# --- LODES: 15-character block GEOID as string; keep Philadelphia blocks (prefix 42101) ---
# CNS (industry) and CE (earnings) are separate marginals: never filter "low-wage × industry".

In [ ]:
# --- LODES OD: h_geocode / w_geocode as 15-character strings; keep pairs with both ends in Philadelphia ---
# Pairs with either end outside the city are kept aside only to compute the cross-boundary share.

In [ ]:
# --- GTFS: convert times past 24:00 (e.g. 25:30:00) to minutes after the service-day midnight ---
# stop_times["dep_s"] = stop_times.departure_time.map(gtfs.time_to_seconds)

In [ ]:
# --- GTFS: select service_ids for the representative weekday AND the previous day ---
# 00:00–05:00 trips usually belong to the previous day's service_id; querying only one day misses most Owl trips.
# ids_today = gtfs.active_service_ids(calendar, calendar_dates, config.GTFS_SERVICE_DATE)
# ids_prev  = gtfs.active_service_ids(calendar, calendar_dates, <previous day>)

In [ ]:
# --- Streetlights: project to EPSG:2272 and snap each pole to the nearest edge within 20 m ---
# Poles beyond the snap distance are dropped and counted.
# poles = lighting.snap_to_edges(poles.to_crs(config.CRS), edges, config.m_to_ft(config.POLE_SNAP_MAX_M))

In [ ]:
# --- 311: filter streetlight outages, compute durations, merge duplicates ---
# 1. Keep the streetlight-outage category (verify name).
# 2. Duration = closed − reported; open cases are capped at 2026-10-31.
# 3. Reports within 15 m with overlapping time are merged into one outage.
# 4. If closure reasons allow, keep only "repaired" closures (drop referrals and duplicates).
# outages = lighting.outage_durations(outages)
# outages = lighting.dedup_outages(outages, config.OUTAGE_DEDUP_M)

## 5. Walk Network (Step 00)

One pedestrian network is built once and shared by every later step; the node IDs and `edge_id` values defined
here are reused through step 06. Outputs: `OUT["00_network"]`.

### 5.1 Download and Largest Connected Component

OSMnx downloads the walk network for Philadelphia buffered by about 1.6 km, so nodes near the city edge can still
reach stops beyond it. Only the largest connected component is kept. Edge lengths stay in metres; geometry is
also projected to EPSG:2272 for spatial joins.

In [ ]:
# Download the OSM walk network for the buffered city boundary and keep the largest connected component
# nodes, edges = network.build_walk_network()

In [ ]:
# Project node and edge geometry to EPSG:2272; keep the edge length column in metres
# nodes, edges = nodes.to_crs(config.CRS), edges.to_crs(config.CRS)

### 5.2 Pandana Network and Edge IDs

Every edge gets a stable `edge_id` (used to attach lighting in step 05a and to accumulate flows in step 05b), and a
pandana `Network` is built for fast shortest-path queries in steps 04–06.

In [ ]:
# Assign a stable edge_id to every edge
# edges["edge_id"] = range(len(edges))

In [ ]:
# Build the pandana Network from node_x, node_y, edge_from, edge_to, edge_length_m
# net = network.to_pandana(nodes, edges)

In [ ]:
# Save nodes and edges as GeoParquet (EPSG:2272)
# nodes.to_parquet(OUT["00_network"]["nodes"]); edges.to_parquet(OUT["00_network"]["edges"])

### 5.3 City Boundary and Tract → PUMA Crosswalk

The city boundary clips later layers, and the tract → PUMA crosswalk supports the consistency check in step 02.

In [ ]:
# Save the Philadelphia boundary
# boundary.to_parquet(OUT["00_network"]["boundary"])

In [ ]:
# Build the tract → PUMA crosswalk from the Census relationship file (GEOID and PUMA as strings)
# crosswalk.to_csv(OUT["00_network"]["tract_puma"], index=False)

### 5.4 Validation

Node and edge counts should be plausible, the network should be a single component, and no edge should have a
zero or negative length.

In [ ]:
# Check counts, a single connected component, and strictly positive edge lengths
# assert (edges.length_m > 0).all()

In [ ]:
# Overview map of the walk network for the website

## 6. Night-Shift Commuter Profile (Step 01)

PUMS microdata describe who commutes at night compared with daytime commuters, and produce the citywide parameters
used in later steps. All statistics are weighted, with replicate-weight standard errors. Outputs: `OUT["01_profile"]`.

### 6.1 Profile Table

Night-shift and daytime commuters are compared on income, car-free rate, industry mix, commute mode, sex, age and
race/ethnicity.

In [ ]:
# Define night-shift vs daytime commuter groups from the decoded departure times

In [ ]:
# Compute weighted shares / medians for each characteristic, with replicate-weight standard errors
# profile = ...; pums.replicate_se(...)

In [ ]:
# Save the profile table and draw the comparison bar charts
# profile.to_csv(OUT["01_profile"]["profile"], index=False)

### 6.2 Citywide Parameters

`p_evening` is the weighted share of "4:00–11:59 p.m." departures that leave between 16:00 and 21:00; it corrects
B08302_015 in step 02. The night transit share and the car-free multiplier (night-shift car-free rate ÷ all-commuter
car-free rate) are reported in the text only, not mapped.

In [ ]:
# p_evening: share of 4:00–11:59 p.m. departures that fall in 16:00–21:00

In [ ]:
# Night-shift transit share and car-free multiplier

In [ ]:
# Write scalar parameters to JSON
# json.dump(params, open(OUT["01_profile"]["params"], "w"))

### 6.3 Industry Night Shares

For workers employed in Philadelphia, the night share of industry k is the weighted share who either arrive at work
between 00:00 and 04:59 (overnight) or leave for work between 16:00 and 21:00 (evening). Small industries are shrunk
toward the citywide share: p̃ₖ = (nₖ·pₖ + m·p_all) ÷ (nₖ + m), with m = 30. The same shares are also computed for
the three LODES OD industry groups (SI01 = CNS01, 02, 04, 05; SI02 = CNS03, 06, 07, 08; SI03 = CNS09–20) for the OD
check in step 04b.

In [ ]:
# Keep workers with POWPUMA in Philadelphia (including work-from-home) and flag night work
# night = JWAP in 00:00–04:59  OR  JWDP in 16:00–21:00   (mutually exclusive, so the flags add)

In [ ]:
# Raw share by CNS: p_k = Σ(PWGTP × night) / Σ PWGTP; n_k = unweighted record count

In [ ]:
# Shrink toward the citywide share: p_tilde = (n_k * p_k + SHRINK_M * p_all) / (n_k + SHRINK_M)

In [ ]:
# Optional: standard error of p_k from the 80 replicate weights; save CNS, n, p, p_tilde, se

In [ ]:
# Same definition and shrinkage for the three SI groups; save 01_si_night_share.csv

### 6.4 Night Departures by PUMA

Weighted night departures by residence PUMA, used for the consistency check in step 02.

In [ ]:
# Weighted night-shift departures per Philadelphia PUMA
# puma_night.to_csv(OUT["01_profile"]["puma_night"], index=False)

### 6.5 Validation

All 20 CNS sectors should have a value, every p̃ₖ should lie between 0 and 1, and accommodation & food (CNS18) and
transportation & warehousing (CNS08) should sit clearly above the citywide average.

In [ ]:
# Check coverage of all 20 sectors, the 0–1 range, and the expected high-night sectors

## 7. Placing Commuters on the Network (Step 02)

Tract and block counts of night-shift commuters are spread onto network nodes so every commuter has a starting
point for routing. Home and work ends share one allocation function. Outputs: `OUT["02_place"]`.

### 7.1 Allocation Function

Tract counts are split to blocks by 2020 population share, then each block's count is divided evenly among all
nodes within 15 m of the block. Blocks with no node inside the buffer fall back to the node nearest their centroid.

In [ ]:
# Block → node lookup: nodes within a 15 m buffer of each block, with nearest-node fallback
# lookup = network.block_node_lookup(blocks, nodes, config.m_to_ft(config.BLOCK_NODE_BUFFER_M))

In [ ]:
# Tract → block (population share) and block → node (even split) allocation
# block_vals = network.tract_to_blocks(tract_counts, block_pop)
# node_vals  = network.blocks_to_nodes(block_vals, lookup)

### 7.2 Home End

Tract count = B08302_002 + B08302_015 × p_evening. Overnight and evening commuters are kept in separate columns.

In [ ]:
# Tract-level night-shift residents: overnight and evening columns

In [ ]:
# Allocate to nodes and save node_id, w_overnight, w_evening, w_total
# home_nodes.to_parquet(OUT["02_place"]["home_nodes"])

### 7.3 Work End

Block count = Σₖ LODES_CNSk × p̃ₖ; this is already at block level, so the tract step is skipped. `work_weights` takes
the share table as an argument so step 06c can swap in alternatives: PUMS all-industry (main), ATUS-weighted,
the original three sectors (CNS08, 16, 18) unweighted, and PUMS-weighted without CNS16.

In [ ]:
# work_weights(lodes, shares) -> Series of night-shift jobs per block
# def work_weights(lodes, shares): ...

In [ ]:
# Compute block weights with the main proxy, allocate to nodes, and save
# work_nodes.to_parquet(OUT["02_place"]["work_nodes"])

In [ ]:
# Also save block-level work-end counts (block_geoid, w_night) for scaling OD flows in step 04b
# work_blocks.to_parquet(config.PROCESSED / "02_work_blocks.parquet")

### 7.4 Bivariate Layer

Tract night-departure density is crossed with the B08141 car-free share (tertiles of each). The legend says
"overlap", not "car-free night-shift commuters", because the two tables cannot be cross-tabulated.

In [ ]:
# Tertiles of night-departure density and car-free share; 3 × 3 bivariate class per tract
# tract_biv.to_parquet(OUT["02_place"]["tract_bivariate"])

### 7.5 PUMA Consistency Check

ACS night departures aggregated to PUMAs are compared with the PUMS estimates from step 01. If PUMA codes do not
match, the comparison falls back to the citywide total.

In [ ]:
# Aggregate B08302 night departures to PUMA and compare with 01 PUMA estimates
# check.to_csv(OUT["02_place"]["consistency"], index=False)

### 7.6 Validation

Node weights must sum to the tract (or block) input within 0.1%, with no missing values; the share of blocks using
the nearest-node fallback is recorded.

In [ ]:
# Check totals within 0.1%, no NaN, and report the fallback share

## 8. Night Service (Step 03)

The stop × route table built here is the only service input for steps 04 and 06. Night service is defined from the
timetable (at least 3 trips between 00:00 and 05:00), not from SEPTA's official Owl list. Outputs: `OUT["03_service"]`.

### 8.1 Night-Window Departures

Departures between 00:00 and 05:00 on the representative weekday are collected, including after-midnight trips that
belong to the previous service day.

In [ ]:
# Active service_ids for the representative weekday and the previous day

In [ ]:
# Keep stop_times departing in the 00:00–05:00 window on the representative date

### 8.2 Stop × Route Headways and Trip Threshold

Each stop × route pair is counted separately (routes at the same stop are not combined): headway = 300 ÷ n_trips.
Pairs with fewer than 3 trips in the window (average headway above 100 minutes) are dropped.

In [ ]:
# Trips and headway per stop × route
# stop_routes = gtfs.stop_route_headways(stop_times, trips, ...)

In [ ]:
# Distribution of stop × route pairs and routes by trip count (1, 2, ≥ 3)

In [ ]:
# Apply the threshold (n_trips >= 3); report dropped routes and their stop counts, and the retained route list
# Stop and report if B1 OWL or L1 OWL is dropped, or far fewer routes remain than expected.

### 8.3 Departure Lists and Longest Gap

For every retained stop × route, the list of departure times is saved for the "actual timetable" alternative in step
06c, and the longest real gap is reported to show how far it can exceed the average headway.

In [ ]:
# Departure minutes per retained stop × route (stop_id, route_id, dep_min)

In [ ]:
# Longest gap between consecutive departures per stop × route (max_gap_min)

### 8.4 Daytime Stops

Stops with weekday service between 07:00 and 19:00, used for the 800 m coverage comparison in step 04.

In [ ]:
# Stops served on the representative weekday between 07:00 and 19:00
# day_stops.to_parquet(OUT["03_service"]["day_stops"])

### 8.5 Validation

B1 OWL and L1 OWL must be present, every headway must be ≤ 100 minutes, and stops must fall within Philadelphia.

In [ ]:
# Check B1 OWL / L1 OWL presence, headway ≤ 100 min, stops inside the city boundary; save outputs

## 9. Access and Coverage (Step 04)

Each weighted node is assigned the stop and route that minimise walking time plus schedule delay; the same function
is re-run for every scenario in step 06. Network-based coverage of daytime and Owl stops is compared alongside.
Outputs: `OUT["04_access"]`.

### 9.1 Stop and Route Choice

For each node, the nearest three Owl stops within 1,600 m of network distance are expanded to all routes serving them.
Cost = walk_m ÷ 1.3 m/s ÷ 60 + 0.5 × headway (with an optional cap), and the lowest-cost stop + route is chosen.
All candidates are also saved, since step 04b uses each node's full candidate route set.

In [ ]:
# Rebuild the pandana network from 00 outputs and attach Owl stops to their nearest nodes

In [ ]:
# Nearest 3 Owl stops within 1,600 m of each node (pandana nearest_pois)

In [ ]:
# Expand candidate stops to every route serving them; cost = walk minutes + schedule delay(headway)

In [ ]:
# Keep the lowest-cost stop + route per node: node_id, stop_id, route_id, walk_m, walk_min, sched_delay_min, access_min
# home_access = access.run_access(net, home_nodes, stop_routes, ...)
# work_access = access.run_access(net, work_nodes, stop_routes, ...)

In [ ]:
# Save every candidate (node_id, stop_id, route_id, cost_min, is_chosen) for home and work ends
# Uses an option on the existing choice function to return candidates; the choice logic is unchanged.
# cand_home.to_parquet(config.PROCESSED / "04_candidates_home.parquet")
# cand_work.to_parquet(config.PROCESSED / "04_candidates_work.parquet")

### 9.2 Nodes Without Service

Nodes with no Owl stop within 1,600 m are flagged `no_service` rather than given an access time of 0.

In [ ]:
# Flag no_service nodes; report node counts and weighted commuters affected (home and work)

### 9.3 Coverage: Daytime vs Owl Stops

800 m network-distance catchments (not circular buffers) are drawn for daytime and Owl stops, and the share of home-
and work-end night-shift commuters inside each is reported.

In [ ]:
# 800 m network catchments for daytime and Owl stops
# access.coverage(net, demand, stop_nodes, config.COVERAGE_RADIUS_M)

In [ ]:
# Share of home and work night-shift commuters covered, day vs night; save coverage table and catchments

### 9.4 Detour Ratio

Detour ratio = network distance ÷ straight-line distance to the chosen stop; high values mark hotspots where the
street grid makes stops harder to reach.

In [ ]:
# Detour ratio per node and hotspot map
# ratio = access.detour_ratio(network_m, euclid_m)

### 9.5 Validation

`access_min` should never be negative, `no_service` counts are reported, and a sample of network distances is
checked against networkx.

In [ ]:
# Check non-negative access_min; spot-check random nodes against networkx shortest paths

## 10. OD Connectivity Check (Step 04b)

Step 04 measures access to the nearest usable Owl route, not to where the worker is going. This step uses LODES
origin–destination flows as a route-topology check: for each night-shift commuting flow, can a candidate route at the
home end reach the work end directly or with one transfer? Timetable coordination and in-vehicle time are not
modelled, and the efficiency and safety definitions are unchanged; the step only adds a set of connectivity measures.
Outputs: `04b_*`.

### 10.1 Night-Shift OD Flows

Only pairs with both ends in Philadelphia are kept. Night flow = SI01·p_SI01 + SI02·p_SI02 + SI03·p_SI03. Each work
block is then scaled so its total inflow equals the step 02 work-end count: OD supplies the destinations, step 02
supplies the totals. Flows are split to home nodes with the step 02 block → node allocation.

In [ ]:
# Night-shift flow per OD pair: SI01–SI03 job counts × p_SI from 01_si_night_share.csv

In [ ]:
# Scale each work block so that Σ_h flow = w_night from 02_work_blocks.parquet
# Count and report work blocks with no OD records.

In [ ]:
# Split each home block's flow evenly across its nodes (step 02 lookup) → home_node × work_block flows

In [ ]:
# Cross-boundary shares: Philadelphia jobs held by residents of other PA counties (main) and of other
# states (WAC − main); Philadelphia residents working elsewhere in PA (main)
# outside.to_csv(config.PROCESSED / "04b_outside_city.csv", index=False)

### 10.2 Route Sets and Transfer Table

R(h) is the home node's candidate route set; R(w) is the union of candidate routes over all nodes in the work block.
Two routes count as transferable if they share a `stop_id` (or `parent_station`), or each has a stop within 100 m
straight-line distance of the other.

In [ ]:
# Route sets: R(h) per home node from 04_candidates_home; R(w) per work block as the union over its nodes

In [ ]:
# Transfer table route_adj from the step 03 stop × route table:
# shared stop_id / parent_station, or stops within TRANSFER_RADIUS_M (convert with config.m_to_ft for geometry)
# route_adj.to_csv(config.PROCESSED / "04b_route_adj.csv", index=False)

In [ ]:
# Check whether B1 OWL and L1 OWL are transferable; stop and report if they are not

### 10.3 Connectivity Classification

Each flow is assigned the first class it satisfies. Direction and timetable are not checked.

| Class | Rule |
| --- | --- |
| E | either end is `no_service` |
| A | the home node's chosen route is in R(w) |
| B | R(h) ∩ R(w) is not empty (another candidate route reaches the work end directly) |
| C | some r1 ∈ R(h) and r2 ∈ R(w) form a pair in `route_adj` (one transfer) |
| D | none of the above |

In [ ]:
# od_connectivity(od_flows, cand_home, cand_work, route_adj, cfg) -> DataFrame(home_node, w, flow, category)
# Re-used for every scenario in step 06.

In [ ]:
# Citywide class distribution; stop and report with examples if class D is very large (e.g. over half)
# or class A is very low; do not tune parameters

### 10.4 Tract-Level Reporting

LODES home locations are noise-infused synthetic data and unreliable at block level, so results are reported and
mapped by home tract only.

In [ ]:
# Save flows with class: home_node, home_tract, w, flow, category → 04b_od_flows.parquet

In [ ]:
# Flow and share per class by home tract, including the A+B+C share → 04b_tract_connectivity.parquet; map

In [ ]:
# Citywide class shares → 04b_summary.csv

### 10.5 Validation

Class flows must add up to the scaled total night flow, class A must agree with the routes chosen in step 04, and a
known home–work pair is checked by hand.

In [ ]:
# Class flows sum to the scaled total (within 0.1%); class A agrees with step 04 chosen routes

In [ ]:
# Spot check: a pair living along L1 OWL and working at a South Philadelphia hospital should be class C;
# print the shared or adjacent stops of the two routes

## 11. Lighting (Step 05a)

Lighting is measured as the effective density of working streetlights per 100 m of edge, net of the lamp-days lost
to reported outages. Outputs: `OUT["05a_lighting"]`.

### 11.1 Pole Snapping and Outage Lamp-Days

Poles are attached to the nearest edge within 20 m, and the de-duplicated outages are converted into lamp-days lost
per edge.

In [ ]:
# Snap poles to edges (20 m max) and count poles per edge_id

In [ ]:
# Attach de-duplicated outages to poles / edges and sum outage lamp-days per edge_id

### 11.2 Effective Density and Poorly Lit Edges

Effective density = (poles × 365 − outage lamp-days) ÷ 365 ÷ edge length, expressed per 100 m. Edges in the lowest
25% are classed as poorly lit.

In [ ]:
# Effective lighting density per 100 m
# eff = lighting.in_service_density(n_poles, outage_lamp_days, length_m)

In [ ]:
# Poorly lit edges: bottom POORLY_LIT_QUANTILE (25%); save edge_id, n_poles, outage_lamp_days, eff_density, is_dark

### 11.3 Stop Lighting

A stop's lighting is the length-weighted mean effective density of all edges within 50 m (tentative radius, kept as a
parameter). Stops in the lowest 25% are classed as poorly lit.

In [ ]:
# Length-weighted effective density of edges within 50 m of each stop; flag the bottom 25%
# stop_light = lighting.stop_lighting(stops, edges, edge_light, ...)

### 11.4 Validation

Report the share of poles that could not be snapped, the outage count before and after de-duplication, and the
quantile cut-off values.

In [ ]:
# Unsnapped pole share, outage counts before/after dedup, cut-off values for edges and stops

## 12. Corridors (Step 05b)

Each node's shortest walk to its chosen stop is traced, commuter weights are added to every edge on the path, and the
poorly lit share of each walk is measured. Home and work ends are computed separately. Outputs: `OUT["05b_corridors"]`.

### 12.1 Shortest Paths

`trace_corridors(net, choice_df, node_weights, edge_lighting, stop_lighting, cfg)` returns edge flows and node
safety metrics, and is re-used for every scenario in step 06.

In [ ]:
# Batch shortest paths from each node to its chosen stop node (pandana shortest_paths)
# paths = corridors.shortest_paths(net, choice.node_id, choice.stop_node)

### 12.2 Edge Flows

Each node's commuter weight is added to every edge on its path, giving night-shift walking flows per edge.

In [ ]:
# Accumulate node weights onto path edges
# edge_flow = corridors.accumulate_edge_flows(paths, weights, edges)

### 12.3 Node Safety Metrics

For each node: metres of the walk on poorly lit edges, and whether the chosen stop is poorly lit.

In [ ]:
# Poorly lit walking distance (m) and poorly lit stop flag per node, for home and work ends
# safety = corridors.commuter_safety(paths, edges, edge_light, stop_light)

### 12.4 Validation

Every path length must equal the step 04 `walk_m` within 1 m.

In [ ]:
# Compare traced path length with walk_m from step 04 (tolerance 1 m)

## 13. Service-Reduction Scenarios (Step 06)

Each scenario edits only the stop × route table; stop/route choice (with candidates), the OD connectivity check and
corridor tracing are then re-run and compared node by node with the baseline. Outputs: `OUT["06_scenarios"]`,
`06_od_results.parquet` and `data/web/`.

### 13.1 Scenario Definitions

| Scenario | Change to the stop × route table |
| --- | --- |
| Baseline | none |
| Headway × 2 | every `headway_min` × 2 |
| Drop B1 OWL | remove B1 OWL rows |
| Drop L1 OWL | remove L1 OWL rows |

In [ ]:
# Build each scenario's stop × route table and report rows / routes remaining
# for name, kw in config.SCENARIOS.items(): scenarios.apply_scenario(stop_routes, **kw)

### 13.2 Re-run Loop

For each scenario and each end (home, work), stop/route choice and corridor tracing are re-run and stacked into one
long table (scenario × end × node).

In [ ]:
# Loop over scenarios × ends; run access (with candidates) and corridors; stack into one long table
# results = scenarios.run_scenario(name, net, stop_routes, demand, ...)

In [ ]:
# For each scenario, re-run od_connectivity on the new candidates; stack into 06_od_results.parquet
# (scenario × home_node × w × category)

### 13.3 Node-Level Changes

Changes are measured relative to the baseline for each node, separately for efficiency and safety. Whether the node
switched stop or route is also recorded.

#### 13.3.1 Efficiency

Change in `access_min`; nodes that move from served to `no_service` are flagged separately.

In [ ]:
# Δ access_min vs baseline; flag served → no_service; flag stop / route switches
# deltas = scenarios.compare_to_baseline(results)

#### 13.3.2 Safety

Change in poorly lit walking distance, and whether the newly chosen stop is poorly lit.

In [ ]:
# Δ poorly lit walking distance (m); new stop poorly lit flag

#### 13.3.3 Connectivity

Change in OD connectivity class per flow. These results are reported alongside Δ`access_min` but do not enter the
regression in step 06b.

In [ ]:
# Class shares per scenario; flows that fall from A/B/C into D or E, by home tract and by removed route

### 13.4 Aggregation

Node-level changes are summarised at three levels: commuters, stops and neighborhoods (tracts).

#### 13.4.1 By Commuter

Weighted mean and distribution of each change across night-shift commuters.

In [ ]:
# Weighted mean and distribution of Δ access_min and Δ poorly lit distance, by scenario and end

#### 13.4.2 By Stop

How many baseline users each stop loses under each scenario.

In [ ]:
# Users per stop in baseline vs scenario; users lost per stop

#### 13.4.3 By Tract

Node results summed by tract for maps and for the association test in step 06b.

In [ ]:
# Aggregate node results to tracts (weighted); save tract results and summary

### 13.5 Validation

Baseline results must match steps 04 and 05b exactly, and in "Drop B1 OWL" only nodes that originally chose B1 may
change.

In [ ]:
# Baseline == step 04 / 05b outputs; in drop-B1, changed nodes ⊆ nodes that chose B1 OWL

In [ ]:
# Baseline OD classes == step 04b; in drop-B1, only flows whose candidate sets include B1 OWL change

## 14. Association Test (Step 06b)

Does the cost of a service cut fall disproportionately on highly dependent (car-free) neighborhoods? A grouped
binomial GLM relates the share of each tract's night-shift residents who are affected to its car-free share.
OD connectivity results (step 04b) are not part of the model. Outputs: `06b_or_table.csv`, `06b_diagnostics.csv`.

### 14.1 Tract Table and "Affected" Definitions

Unit: tract; nₜ = night-shift residents, yₜ = those affected. Tracts with no Owl stop within 1,600 m at baseline are
reported separately and excluded from the model. All thresholds are config parameters.

| Scenario × dimension | Affected if |
| --- | --- |
| Drop B1 / Drop L1 (efficiency) | chosen route removed and Δ ≥ 10 min |
| Headway × 2 (efficiency) | new access time > 45 min |
| All scenarios (safety) | poorly lit walking +≥ 100 m, or new stop poorly lit |

In [ ]:
# Build the tract table: n_night, n_affected, p_affected, car-free quartile; drop tracts without baseline service

### 14.2 Binomial GLM

Grouped binomial GLM with logit link and `var_weights = n`; the car-free share enters as quartiles with the lowest
quartile as reference. Results are reported as odds ratios with 95% confidence intervals.

In [ ]:
# run_regression(tract_df, cfg) -> or_table
# m = smf.glm("p_affected ~ C(zv_q, Treatment(1))", data=tr,
#             family=sm.families.Binomial(), var_weights=tr.n_night).fit()
# or_table = np.exp(pd.concat([m.params, m.conf_int()], axis=1))

### 14.3 Supplementary Models

The car-free quartile is replaced in turn by poverty rate, Black share and Hispanic share, one at a time; VIF is
computed first.

In [ ]:
# VIF of candidate predictors; fit one supplementary model per predictor

### 14.4 Diagnostics

Spatial autocorrelation in Pearson residuals is checked with Moran's I (queen contiguity). If complete separation
occurs, a descriptive risk ratio (affected share in the high-dependency group ÷ low-dependency group) is reported
instead.

In [ ]:
# Moran's I of Pearson residuals (esda, queen weights); separation check and risk-ratio fallback; save diagnostics

## 15. Sensitivity Analysis (Step 06c)

Key parameters are changed one at a time (OAT) while the others stay at baseline, and the pipeline is re-run from the
earliest affected step. Outputs: `06c_results.csv`, `06c_hotspot_overlap.csv`, `06c_tornado_*.png`.

### 15.1 Parameter Grid and Run Loop

| Parameter | Baseline | Alternatives | Re-run from |
| --- | --- | --- | --- |
| Schedule delay | 0.5 × headway | capped at 60 min; 1.0 × headway; median wait from actual timetable (also 90th pct.) | 04 |
| Work-end proxy | PUMS all-industry | ATUS-weighted; 3 sectors unweighted; without CNS16 | 02 (work end) |
| Poorly lit threshold | bottom 25% | 10%; 33% | 05a / 05b |
| "Affected" thresholds | 10 min / 45 min / 100 m | 5, 15 min / 30, 60 min / 50, 200 m | 06b |

Four outputs are tracked: home-end weighted mean access time (baseline), Δ per scenario, weighted mean poorly lit
walking distance, and the main model's top-vs-bottom quartile odds ratio.

In [ ]:
# One-at-a-time loop over SENS_GRID
# for name, alts in SENS_GRID.items():
#     for v in alts:
#         cfg = replace(BASE, **{name: v})
#         results.append(run_steps_2_to_6b(cfg) | {"param": name, "value": v})

### 15.2 Tornado Plots

One tornado plot per tracked output shows the range produced by each parameter.

In [ ]:
# Tornado plot for each of the four tracked outputs; save 06c_tornado_*.png

### 15.3 Hotspot Stability and Robustness Rule

For each run, the top 10% most-affected tracts are compared with the baseline by Jaccard index, and full rankings by
Spearman correlation. A result is considered robust if Jaccard ≥ 0.6 and the severity order of the three scenarios
is unchanged.

In [ ]:
# Jaccard of top-10% tracts and Spearman rank correlation vs baseline; apply robustness rule; save overlap table

## 16. Web Export

Lightweight files for the website are written to `data/web/` (site technology still undecided). Everything is
reprojected to EPSG:4326, reduced to display fields, and simplified; `site/` and `dashboard/` are not modified.

### 16.1 Layer List

- Tract bivariate layer; night-shift job blocks
- Owl stops (routes, headway, lighting) and 800 m catchments
- Corridors (edge flow + lighting; edges with flow > 0 only)
- Tract results per scenario, and stop changes
- Tract OD connectivity (class shares, A+B+C share), baseline and per scenario
- Profile bar-chart data, odds-ratio table, tornado data

### 16.2 Export

Each layer is reprojected, trimmed to display fields, simplified, and written as GeoJSON, JSON or CSV.

In [ ]:
# Helper: reproject to EPSG:4326, keep display columns, simplify geometry, write GeoJSON
# def to_web(gdf, cols, name, tol_ft=...): ...

In [ ]:
# Map layers: tract bivariate, job blocks, Owl stops + catchments, corridors

In [ ]:
# Scenario layers and chart data: tract results, stop changes, profile, OR table, tornado data

## 17. Limitations

A summary of known limitations, to be expanded as results come in:

- *Access is measured to the nearest usable Owl route, not to the worker's destination. A topological check with
  LODES origin–destination flows shows whether candidate routes connect each home–work pair directly or with one
  transfer, but it does not test direction, timetable coordination, or in-vehicle time, and it assumes night workers
  share the commuting pattern of all workers in the same industry group.*
- Philadelphia residents who work in New Jersey or Delaware are not included in the OD check.
- PUMS uses Pennsylvania residents only, so industry night shares leave out out-of-state commuters
  (13% of Philadelphia jobs in LODES 2024).
- Closed 311 cases do not always mean the light was repaired.
- Average headway understates waiting risk where real gaps are much longer (see `max_gap_min`).
- B08302 and B08141 cannot be cross-tabulated; the bivariate map shows overlap only.
- Industry night shares are estimated at PUMA level and applied uniformly to every block.
- ATUS shares are national; only spatial patterns and rankings are compared.
- One representative weekday stands in for all weekday service.
- The 50 m stop-lighting radius is provisional.

## Appendix A. Parameters

Names follow `config.py`; parameters marked *to add* are not yet in the file.

| Parameter | Value | Used in |
| --- | --- | --- |
| `NIGHT_START_S` / `NIGHT_END_S` | 00:00 / 05:00 | all |
| `EVENING_DEPART` | 16:00–21:00 | 01, 02 |
| `BLOCK_NODE_BUFFER_M` | 15 | 02 |
| `SHRINK_M` (*to add*) | 30 | 01 |
| `WORK_PROXY` (*to add*) | `"pums_all"` (alt.: `"atus"`, `"three_sectors"`, `"no_health"`) | 02, 06c |
| `GTFS_SERVICE_DATE` | representative weekday, to verify | 03 |
| `OWL_MIN_TRIPS` | 3 | 03 |
| `SCHEDULE_DELAY_FACTOR` | 0.5 | 04, 06 |
| `SCHEDULE_DELAY_CAP` (*to add*) | None (alt. 60) | 04, 06c |
| `WALK_SPEED_MPS` | 1.3 | 04 |
| `COVERAGE_RADIUS_M` / `SEARCH_RADIUS_M` | 800 / 1600 | 04 |
| `N_CANDIDATE_STOPS` | 3 | 04 |
| `OD_JOB_TYPE` (*to add*) | `"JT00"` | 04b |
| `TRANSFER_RADIUS_M` (*to add*) | 100 | 04b |
| `OD_REPORT_LEVEL` (*to add*) | `"tract"` | 04b, 06 |
| `POORLY_LIT_QUANTILE` | 0.25 | 05a, 05b |
| `STOP_LIGHTING_RADIUS_M` | 50 (provisional) | 05a |
| `POLE_SNAP_MAX_M` | 20 | 05a |
| `OUTAGE_WINDOW` | 2025-11-01 to 2026-10-31 | 05a |
| `OUTAGE_DEDUP_M` | 15 | 05a |
| `SCENARIOS` | baseline, headway × 2, drop B1 OWL, drop L1 OWL | 06 |
| `AFFECT_DROP_MIN` / `AFFECT_X2_MIN` / `AFFECT_DARK_M` (*to add*) | 10 / 45 / 100 | 06b |
| `SENS_GRID` (*to add*) | see Section 15.1 | 06c |
| `HOTSPOT_TOP` / `JACCARD_ROBUST` (*to add*) | 0.10 / 0.6 | 06c |
| `CRS` | EPSG:2272 | all |

## Appendix B. NAICS → LODES CNS Crosswalk and ATUS Shares

| CNS | NAICS | Industry | CNS | NAICS | Industry |
| --- | --- | --- | --- | --- | --- |
| 01 | 11 | Agriculture, forestry, fishing & hunting | 11 | 53 | Real estate & rental |
| 02 | 21 | Mining | 12 | 54 | Professional & technical services |
| 03 | 22 | Utilities | 13 | 55 | Management of companies |
| 04 | 23 | Construction | 14 | 56 | Administrative support & waste management |
| 05 | 31–33 | Manufacturing | 15 | 61 | Educational services |
| 06 | 42 | Wholesale trade | 16 | 62 | Health care & social assistance |
| 07 | 44–45 | Retail trade | 17 | 71 | Arts, entertainment & recreation |
| 08 | 48–49 | Transportation & warehousing | 18 | 72 | Accommodation & food services |
| 09 | 51 | Information | 19 | 81 | Other services |
| 10 | 52 | Finance & insurance | 20 | 92 | Public administration |

ATUS non-daytime shares, BLS USDL-26-1533 Table 7 (2024–2025): CNS01 0 (suppressed), 02 0 (suppressed), 03 0.245,
04 0.031, 05 0.153, 06–07 0.200, 08 0.245, 09 0.087, 10–11 0.035, 12–14 0.075, 15–16 0.110, 17–18 0.400, 19 0.142,
20 0.142. The 2017–2018 values (USDL-19-1691) used in Plotts (2025) Appendix A.1 are kept for comparison; the
post-pandemic release is used because it matches the 2020–2024 ACS/PUMS and 2024 LODES periods.

## Appendix C. Differences from the Proposal

| Aspect | Proposal | Current approach |
| --- | --- | --- |
| Efficiency / safety | Safety included waiting time | Efficiency = walking + schedule delay; safety = poorly lit walking + stop lighting, no waiting |
| Dependency measure | B25044 car-free households | B08141 car-free commuters, as a bivariate overlap |
| Work end | Three night-heavy sectors | All 20 sectors × PUMS industry night shares |
| Poorly lit | Pole density or unrepaired outages | Bottom 25% of effective lighting density |
| Headway | Per stop | Per stop × route |
| Night service | No explicit threshold | ≥ 3 trips 00:00–05:00 per stop × route, independent of the official Owl list |
| New | — | Step 04b OD connectivity check; step 06b logistic regression; step 06c sensitivity analysis |
| Deliverable | Quarto + Panel + three-page report | GitHub Pages site + repository + presentation (site stack undecided) |